In [30]:
2**12

4096

In [1]:
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pylab as plt
from tqdm import tqdm_notebook

In [2]:
device = 'cuda:0'

In [3]:
PATTERN_LEN = 12
SEQ_LEN = 24
TASK_NUM = 1000
SIGMA = 0.1
BS = 2**10


In [30]:
(PATTERN_LEN + 2**(SEQ_LEN - PATTERN_LEN))/4/BS , 2**PATTERN_LEN / TASK_NUM

(1.0029296875, 4.096)

In [4]:
def generate(PATTERN_LEN, SEQ_LEN, TASK_NUM, BS):
    assert (PATTERN_LEN + 2**(SEQ_LEN - PATTERN_LEN))/4/BS > 1
    # количество комбинаций на общий размер датасета должен быть > 1
    assert 2**PATTERN_LEN > TASK_NUM
    
    gold_params = []
    X = []
    Y = []
    VALIDATION_META = []
    
    for _ in tqdm_notebook(range(TASK_NUM)):
        VALIDATION_META.append([0] * (BS * 4))
        pattern = ''.join([
            '01'[i]
            for i in np.random.randint(low=0, high=2, size=(PATTERN_LEN))
        ])

        p = -1 + 2 * torch.tensor(list(map(float, pattern)))
        negative_slope = 1e-6
        scale = 100.0

        gold_params.append(
            torch.nn.Sequential(
                torch.nn.Linear(SEQ_LEN, SEQ_LEN - PATTERN_LEN + 1),
                torch.nn.LeakyReLU(negative_slope),
                torch.nn.Linear(SEQ_LEN - PATTERN_LEN + 1, 1),
            )
        )

        W1 = gold_params[-1][0]
        W2 = gold_params[-1][2]

        W1.weight.data.zero_()
        W1.bias.data.zero_()

        W2.weight.data.zero_()
        W2.bias.data.zero_()

        n_windows = SEQ_LEN - PATTERN_LEN + 1

        for i in range(n_windows):

            W1.weight.data[i, i:i+PATTERN_LEN] = p / scale

            W1.bias.data[i] = -(PATTERN_LEN - 0.5) / scale

        W2.weight.data += scale

        #
        # negative contribution from mismatching windows
        #
        neg = 1.5 * negative_slope

        #
        # positive sample:
        #   0.5 - neg*(n_windows-1)
        #
        # negative sample:
        #   -neg*n_windows
        #
        # choose midpoint
        #
        thr = 0.25 - neg * (n_windows - 0.5)

        W2.bias.data.fill_(-thr)
        x = []
        y = []

        for j in range(BS * 4):

            x.append(''.join([
                '01'[i]
                for i in np.random.randint(low=0, high=2, size=(SEQ_LEN))
            ]))

            y.append(1 * (pattern in x[-1]))

        while sum(y[:BS]) < BS // 2:

            zeros = np.where(np.array(y[:BS]) == 0)[0]

            j = np.random.choice(zeros, 1)[0]

            offset = np.random.randint(
                low=0,
                high=SEQ_LEN - PATTERN_LEN + 1
            )

            x[j] = (
                x[j][:offset]
                + pattern
                + x[j][offset + PATTERN_LEN:]
            )

            y[j] = 1

        while sum(y[BS:2*BS]) < BS // 2:

            zeros = BS + np.where(np.array(y[BS:2*BS]) == 0)[0]

            j = np.random.choice(zeros, 1)[0]

            offset = np.random.randint(
                low=0,
                high=SEQ_LEN - PATTERN_LEN + 1
            )

            x[j] = (
                x[j][:offset]
                + pattern
                + x[j][offset + PATTERN_LEN:]
            )

            y[j] = 1
        
        for i in range(2*BS, 3*BS):
            x[i] = x[i-2*BS][:]
            y[i] = y[i-2*BS]
            
            alternatives = []
            alternatives.append(list(x[i][:]))
            alternatives.append(list(x[i][:]))
            alternatives.append(list(x[i][:]))
            alternatives.append(list(x[i][:]))
            
            alternative_meta = []
            alternative_meta.append(-1)
            alternative_meta.append(-1)
            alternative_meta.append(1)
            alternative_meta.append(1)
          
            
            
            alternatives[0][1:] = x[i][:-1]
            alternatives[0][0] = '0'
            alternatives[1][1:] = x[i][:-1]
            alternatives[1][0] = '1'
            alternatives[2][:-1] = x[i][1:]
            alternatives[2][-1] = '0'
            alternatives[3][:-1] = x[i][1:]
            alternatives[3][-1] = '1'
            
            good_alts = []
            
            for j in range(4):
                y_alt = 0
                if pattern in ''.join(alternatives[j]):
                    y_alt = 1
                if y[i] == y_alt:
                    good_alts.append(j)
            alt_id = np.random.choice(good_alts)
            x[i] = ''.join(alternatives[alt_id])
            VALIDATION_META[-1][i] = alternative_meta[alt_id]
        for i in range(3*BS, 4*BS):
            x[i] = x[i-2*BS][:]
            y[i] = y[i-2*BS]
            alternatives = []
            alternative_meta = []
            alternative_meta.append(-1)
            alternative_meta.append(-1)
            alternative_meta.append(1)
            alternative_meta.append(1)
            
            alternatives.append(list(x[i][:]))
            alternatives.append(list(x[i][:]))
            alternatives.append(list(x[i][:]))
            alternatives.append(list(x[i][:]))
            
            alternatives[0][1:] = x[i][:-1]
            alternatives[0][0] = '0'
            alternatives[1][1:] = x[i][:-1]
            alternatives[1][0] = '1'
            alternatives[2][:-1] = x[i][1:]
            alternatives[2][-1] = '0'
            alternatives[3][:-1] = x[i][1:]
            alternatives[3][-1] = '1'
            
            good_alts = []
            
            for j in range(4):
                y_alt = 0
                if pattern in ''.join(alternatives[j]):
                    y_alt = 1
                if y[i] == y_alt:
                    good_alts.append(j)
            alt_id = np.random.choice(good_alts)
            x[i] = ''.join(alternatives[alt_id])
            VALIDATION_META[-1][i] = alternative_meta[alt_id]
            
            
        x = torch.tensor([
            list(map(int, x[j]))
            for j in range(4 * BS)
        ]).float().to(device)

        y = torch.tensor(y).to(device)

        X.append(2 * x - 1.0)
        Y.append(y)
    return X, Y, gold_params, VALIDATION_META


In [8]:
!mv conv_data_large_1000_.pckl conv_data_large_.pckl

In [5]:
LABEL = ''
np.random.seed(0)
X, Y, gold_params, val_meta = generate(PATTERN_LEN, SEQ_LEN, TASK_NUM, BS)
with open(f'conv_data_large_{LABEL}.pckl', 'wb') as out:
    import pickle
    out.write(pickle.dumps([X, Y, gold_params, val_meta]))

/tmp/ipykernel_2997184/4069676509.py:11: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for _ in tqdm_notebook(range(TASK_NUM)):


  0%|          | 0/1000 [00:00<?, ?it/s]

In [6]:
LABEL = ''


for pseudo_bs in [16, 32, 64, 256, 512]:
    LABEL = f'bs_{pseudo_bs}'
    np.random.seed(42+pseudo_bs)
    X, Y, gold_params, val_meta = generate(PATTERN_LEN, SEQ_LEN, TASK_NUM, BS)
    with open(f'conv_data_large_{LABEL}.pckl', 'wb') as out:
        import pickle
        out.write(pickle.dumps([X, Y, gold_params, val_meta]))

/tmp/ipykernel_2997184/4069676509.py:11: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for _ in tqdm_notebook(range(TASK_NUM)):


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

In [7]:
LABEL = ''

for tn_per_run in [1, 5, 10, 25, 50]:
    tn = tn_per_run * 10 # 5 for train, 5 for val
    LABEL = f'tn_{tn_per_run}'
    np.random.seed(999+tn)
    X, Y, gold_params, val_meta = generate(PATTERN_LEN, SEQ_LEN, tn, BS)
    with open(f'conv_data_large_{LABEL}.pckl', 'wb') as out:
        import pickle
        out.write(pickle.dumps([X, Y, gold_params, val_meta]))

/tmp/ipykernel_2997184/4069676509.py:11: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for _ in tqdm_notebook(range(TASK_NUM)):


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]